# Consolidação interna e comparação entre taxonomias

O fluxo tem dois níveis:

1. **Dentro de A e dentro de B:** identificar e mesclar tópicos semanticamente próximos,
   separadamente, usando o mesmo critério.
2. **Entre A e B:** comparar as duas taxonomias já consolidadas.

Todo o processamento é local, com embeddings; não utiliza API da OpenAI.
O notebook também calcula a comparação antes do merge para mostrar seu efeito nas métricas.

A mesclagem usa **ligação completa**: dois grupos só são unidos quando todos os pares de
tópicos entre eles atingem o limiar interno. Isso evita a cadeia A≈B, B≈C, mas A≠C.

A similaridade é um critério operacional, não uma prova de equivalência temática. Tópicos
de um mesmo domínio podem continuar distintos. Por isso, o limiar de merge é mais exigente
que o limiar de correspondência entre as abordagens, e os grupos são exportados para revisão.
O padrão de 0,95 é conservador: um limiar menor pode unir assuntos distintos que compartilham
muito vocabulário do domínio. Não é obrigatório haver mesclagens; grupos unitários são válidos.

Os IDs originais e os textos de todos os membros são preservados. O rótulo exibido é o do
**medoide**: o tópico mais central do grupo. Na comparação, o grupo é representado pela
média dos embeddings dos membros, normalizada, para considerar todos os textos originais.

Se necessário, instale as dependências no kernel:
    
    %pip install numpy sentence-transformers

Na primeira execução, o modelo de embeddings pode precisar ser baixado.


In [ ]:
from pathlib import Path

ARQUIVO_A = Path("topics_final_order.json")
ARQUIVO_B = Path("topics_final_shuff.json")
NOME_A = "Ordem original"
NOME_B = "Shuffle"

MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DISPOSITIVO = "cpu"
SOMENTE_MODELO_LOCAL = False  # True: usa somente um modelo já baixado.

# Merge dentro de cada abordagem. Aumente para mesclar de forma mais conservadora.
LIMIAR_MESCLAGEM_INTERNA = 0.95
# Representação dos tópicos originais pelos grupos da própria abordagem.
LIMIAR_COBERTURA_INTERNA = 0.5
# Correspondências entre as duas abordagens consolidadas.
LIMIAR_SIMILARIDADE = 0.5

PASTA_SAIDA = Path("outputs/comparacao_taxonomias_consolidadas")


## 1. Métricas dentro de cada abordagem

A consolidação de A não consulta B, e a de B não consulta A.

- **Pares candidatos antes/depois:** pares internos cujo cosseno atinge o limiar de merge.
  São indícios de proximidade; não são uma anotação humana de duplicidade.
- **Taxa de redução:** `1 − número de grupos / número de tópicos originais`.
- **Coesão dos grupos mesclados:** média e mínimo dos cossenos entre seus membros.
  Grupos unitários não entram nesse cálculo.
- **Retenção dos IDs de origem:** proporção dos tópicos originais preservados nos grupos.
  Cada ID precisa aparecer exatamente uma vez.
- **Cobertura semântica interna:** proporção dos tópicos originais que encontram um
  centroide de sua abordagem acima do limiar de cobertura interna.

A retenção dos IDs é uma garantia estrutural. A cobertura semântica é um indicador sobre
embeddings e tende a ser alta neste procedimento, pois os grupos são construídos a partir
dos próprios tópicos. **Nenhuma dessas medidas representa a cobertura dos 1.800 vídeos**:
isso exigiria classificar todos os títulos na taxonomia.

Com similaridades empatadas, o desempate usa os textos e IDs, evitando depender da posição
das linhas no arquivo. O agrupamento completo é conservador e pode preservar grupos ainda
relacionados entre si quando nem todos os pares de seus membros passam no limiar.

Referência para ligação completa:
[SciPy — hierarchical linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html).


## 2. Métricas entre as abordagens consolidadas

Se \(C_{ij}\) é o cosseno entre o tópico \(i\) de A e o tópico \(j\) de B:

\[
S_{A \rightarrow B} = \frac{1}{|A|}\sum_i \max_j C_{ij}, \quad
S_{B \rightarrow A} = \frac{1}{|B|}\sum_j \max_i C_{ij}
\]

\[
S_{\mathrm{global}} = \frac{S_{A \rightarrow B} + S_{B \rightarrow A}}{2}
\]

O cosseno varia de **−1 a 1**; valores próximos de 1 indicam maior semelhança no espaço
dos embeddings. A similaridade global é uma agregação exploratória desses valores,
**não uma porcentagem de acerto**.

A cobertura usa o limiar configurado: um tópico é coberto quando sua melhor similaridade
é maior ou igual a esse valor. O F1 de cobertura varia de 0 a 1. O limiar de 0,5 é um
ponto de partida configurável, não uma equivalência validada entre os temas.

Um mesmo tópico pode corresponder a vários outros. Assim, a similaridade global pode ser
alta mesmo quando uma taxonomia divide um assunto em mais tópicos. A quantidade de tópicos,
a cobertura em cada direção e a matriz ajudam a identificar essa diferença de granularidade.

A comparação utiliza rótulos e descrições. As evidências de criação não são uma classificação
completa dos vídeos, portanto este notebook não calcula ARI ou NMI de agrupamentos.

Referência para os embeddings e o cosseno:
[Sentence Transformers — Semantic Textual Similarity](https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html).


In [ ]:
import csv
import hashlib
import html
import json
from datetime import datetime, timezone

import numpy as np


def carregar_taxonomia(caminho):
    caminho = Path(caminho)
    dados = json.loads(caminho.read_text(encoding="utf-8-sig"))
    if not isinstance(dados, dict):
        raise ValueError(f"{caminho}: esperado um objeto JSON.")
    chave = "topics" if "topics" in dados else "groups"
    entradas = dados.get(chave)
    if not isinstance(entradas, list) or not entradas:
        raise ValueError(f"{caminho}: informe uma lista topics ou groups não vazia.")
    topicos = []
    ids = set()
    for indice, entrada in enumerate(entradas, 1):
        if not isinstance(entrada, dict):
            raise ValueError(f"{caminho}: tópico {indice} inválido.")
        for campo in ("label", "description"):
            if not isinstance(entrada.get(campo), str) or not entrada[campo].strip():
                raise ValueError(f"{caminho}: tópico {indice} sem {campo} válido.")
        identificador = entrada.get("topic_id") if chave == "topics" else f"G{indice:03d}"
        if not isinstance(identificador, (str, int)) or isinstance(identificador, bool) or not str(identificador).strip():
            raise ValueError(f"{caminho}: tópico {indice} sem topic_id válido.")
        identificador = str(identificador)
        if identificador in ids:
            raise ValueError(f"{caminho}: ID duplicado: {identificador}.")
        ids.add(identificador)
        topicos.append({
            "id": identificador,
            "label": entrada["label"].strip(),
            "description": entrada["description"].strip(),
        })
    return topicos


def matriz_cosseno(embeddings_a, embeddings_b):
    a = np.asarray(embeddings_a, dtype=float)
    b = np.asarray(embeddings_b, dtype=float)
    if (a.ndim != 2 or b.ndim != 2 or not a.shape[0] or not b.shape[0]
            or not a.shape[1] or a.shape[1] != b.shape[1]):
        raise ValueError("Embeddings devem ser matrizes não vazias com a mesma dimensão.")
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError("Embeddings contêm valores não finitos.")
    norma_a = np.linalg.norm(a, axis=1, keepdims=True)
    norma_b = np.linalg.norm(b, axis=1, keepdims=True)
    if np.any(norma_a == 0) or np.any(norma_b == 0):
        raise ValueError("Embeddings com norma zero não permitem calcular cosseno.")
    return np.clip((a / norma_a) @ (b / norma_b).T, -1.0, 1.0)


def comparar_topicos(topicos_a, topicos_b, similaridades, limiar=0.5):
    matriz = np.asarray(similaridades, dtype=float)
    if not topicos_a or not topicos_b or matriz.shape != (len(topicos_a), len(topicos_b)):
        raise ValueError("A matriz deve conter uma linha por tópico de A e uma coluna por tópico de B.")
    if not np.isfinite(matriz).all() or np.any(matriz < -1) or np.any(matriz > 1):
        raise ValueError("Cossenos devem ser finitos e estar entre -1 e 1.")
    if not isinstance(limiar, (int, float)) or isinstance(limiar, bool) or not 0 <= limiar <= 1:
        raise ValueError("O limiar deve estar entre 0 e 1.")
    melhores_a = matriz.max(axis=1)
    melhores_b = matriz.max(axis=0)
    cobertura_a = float(np.mean(melhores_a >= limiar))
    cobertura_b = float(np.mean(melhores_b >= limiar))
    media_a = float(melhores_a.mean())
    media_b = float(melhores_b.mean())
    soma_coberturas = cobertura_a + cobertura_b
    metricas = {
        "topics_a": len(topicos_a),
        "topics_b": len(topicos_b),
        "similarity_a_to_b": media_a,
        "similarity_b_to_a": media_b,
        "global_similarity": (media_a + media_b) / 2,
        "coverage_a_to_b": cobertura_a,
        "coverage_b_to_a": cobertura_b,
        "coverage_f1": 2 * cobertura_a * cobertura_b / soma_coberturas if soma_coberturas else 0.0,
        "topics_a_below_threshold": int(np.sum(melhores_a < limiar)),
        "topics_b_below_threshold": int(np.sum(melhores_b < limiar)),
    }
    pares = []
    for direcao, indices in (
        ("A -> B", enumerate(matriz.argmax(axis=1))),
        ("B -> A", ((i, j) for j, i in enumerate(matriz.argmax(axis=0)))),
    ):
        for i, j in indices:
            valor = float(matriz[i, j])
            pares.append({
                "direction": direcao,
                "id_a": topicos_a[i]["id"],
                "label_a": topicos_a[i]["label"],
                "id_b": topicos_b[j]["id"],
                "label_b": topicos_b[j]["label"],
                "cosine_similarity": valor,
                "above_threshold": valor >= limiar,
            })
    return {"metrics": metricas, "threshold": float(limiar), "best_matches": pares}


def criar_html(relatorio, topicos_a, topicos_b, similaridades):
    escapar = html.escape
    nome_a = escapar(relatorio["approach_a"]["name"])
    nome_b = escapar(relatorio["approach_b"]["name"])
    metricas = relatorio["metrics"]
    partes = [
        '<!doctype html><html lang="pt-BR"><meta charset="utf-8">',
        "<title>Comparação de taxonomias</title>",
        "<style>body{font:14px system-ui;margin:24px;color:#172033}"
        ".matriz{overflow:auto}table{border-collapse:collapse}th,td{border:1px solid #ccd5e0;padding:8px}"
        "th{background:#edf2f7;text-align:left}td{text-align:right;font-variant-numeric:tabular-nums}"
        ".nota{max-width:1000px;line-height:1.5}</style>",
        f"<h2>{nome_a} × {nome_b}</h2>",
        f"<p>Similaridade global: <strong>{metricas['global_similarity']:.4f}</strong>"
        f" | Tópicos: {metricas['topics_a']} × {metricas['topics_b']}</p>",
        f"<p>Cobertura A → B: {metricas['coverage_a_to_b']:.1%}"
        f" | Cobertura B → A: {metricas['coverage_b_to_a']:.1%}"
        f" | F1 de cobertura: {metricas['coverage_f1']:.4f}"
        f" | Limiar: {relatorio['threshold']:.2f}</p>",
        '<p class="nota">As células mostram o cosseno entre os textos de cada par de tópicos.'
        " O tom azul indica maior similaridade. Passe o cursor para ver os rótulos."
        " A pontuação não é uma porcentagem de acerto; correspondências podem ser muitos-para-um.</p>",
        '<div class="matriz"><table><thead><tr><th>A / B</th>',
    ]
    for topico in topicos_b:
        partes.append(f'<th title="{escapar(topico["label"], quote=True)}">{escapar(topico["id"])}</th>')
    partes.append("</tr></thead><tbody>")
    for i, topico_a in enumerate(topicos_a):
        partes.append(f'<tr><th>{escapar(topico_a["id"])} — {escapar(topico_a["label"])}</th>')
        for j, topico_b in enumerate(topicos_b):
            valor = float(similaridades[i, j])
            intensidade = max(0.0, valor) * 0.8
            cor_texto = "white" if valor > 0.75 else "#172033"
            titulo = escapar(f'{topico_a["label"]} / {topico_b["label"]}: {valor:.4f}', quote=True)
            partes.append(f'<td title="{titulo}" style="background:rgba(37,99,235,{intensidade:.3f});'
                          f'color:{cor_texto}">{valor:.3f}</td>')
        partes.append("</tr>")
    partes.append("</tbody></table></div></html>")
    return "".join(partes)


def salvar_resultados(relatorio, topicos_a, topicos_b, similaridades, pasta):
    pasta = Path(pasta)
    arquivos = {
        "report": pasta / "comparison_report.json",
        "matrix": pasta / "similarity_matrix.csv",
        "matches": pasta / "best_matches.csv",
        "html": pasta / "comparison.html",
    }
    entradas = {Path(relatorio[chave]["file"]).resolve() for chave in ("approach_a", "approach_b")}
    if any(caminho.resolve() in entradas for caminho in arquivos.values()):
        raise ValueError("O destino de um relatório coincide com um arquivo de entrada.")
    pasta.mkdir(parents=True, exist_ok=True)
    arquivos["report"].write_text(json.dumps(relatorio, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    with arquivos["matrix"].open("w", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(["A / B"] + [t["id"] for t in topicos_b])
        for topico, linha in zip(topicos_a, similaridades):
            escritor.writerow([topico["id"]] + [f"{float(valor):.8f}" for valor in linha])
    with arquivos["matches"].open("w", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=list(relatorio["best_matches"][0]))
        escritor.writeheader()
        escritor.writerows(relatorio["best_matches"])
    arquivos["html"].write_text(criar_html(relatorio, topicos_a, topicos_b, similaridades), encoding="utf-8")
    return arquivos


In [ ]:
from itertools import combinations


def validar_limiar(valor, nome):
    if not isinstance(valor, (int, float)) or isinstance(valor, bool) or not 0 <= valor <= 1:
        raise ValueError(f"{nome} deve ser um número entre 0 e 1.")


def pares_internos(topicos, similaridades, limiar):
    validar_limiar(limiar, "Limiar interno")
    matriz = np.asarray(similaridades, dtype=float)
    if matriz.shape != (len(topicos), len(topicos)) or not np.isfinite(matriz).all():
        raise ValueError("Matriz interna inválida.")
    return [
        {
            "id_1": topicos[i]["id"], "label_1": topicos[i]["label"],
            "id_2": topicos[j]["id"], "label_2": topicos[j]["label"],
            "cosine_similarity": float(matriz[i, j]),
        }
        for i, j in combinations(range(len(topicos)), 2) if matriz[i, j] >= limiar
    ]


def consolidar_semanticamente(topicos, embeddings, limiar=0.95, prefixo="G"):
    validar_limiar(limiar, "Limiar de mesclagem")
    vetores = np.asarray(embeddings, dtype=float)
    if vetores.ndim != 2 or len(topicos) != len(vetores) or not topicos:
        raise ValueError("Informe um embedding para cada tópico.")
    ids = [t["id"] for t in topicos]
    if len(ids) != len(set(ids)):
        raise ValueError("IDs duplicados na taxonomia de origem.")
    similaridades = matriz_cosseno(vetores, vetores)
    vetores = vetores / np.linalg.norm(vetores, axis=1, keepdims=True)

    def chave_topico(indice):
        t = topicos[indice]
        return (t["label"].casefold(), t["description"].casefold(), t["id"])

    def chave_grupo(indices):
        return tuple(sorted(chave_topico(i) for i in indices))

    grupos = sorted([[i] for i in range(len(topicos))], key=chave_grupo)
    while True:
        melhor = None
        for i, j in combinations(range(len(grupos)), 2):
            # Ligação completa: o pior par entre os grupos precisa passar no limiar.
            score = float(similaridades[np.ix_(grupos[i], grupos[j])].min())
            if score < limiar:
                continue
            chave = (-round(score, 12), chave_grupo(grupos[i] + grupos[j]))
            if melhor is None or chave < melhor[0]:
                melhor = (chave, i, j)
        if melhor is None:
            break
        _, i, j = melhor
        uniao = sorted(grupos[i] + grupos[j], key=chave_topico)
        grupos = [g for k, g in enumerate(grupos) if k not in (i, j)] + [uniao]
        grupos.sort(key=chave_grupo)

    consolidados = []
    centroides = []
    for numero, membros in enumerate(grupos, 1):
        membros = sorted(membros, key=chave_topico)
        submatriz = similaridades[np.ix_(membros, membros)]
        medias = submatriz.mean(axis=1)
        candidatos = [membros[k] for k, media in enumerate(medias)
                      if np.isclose(media, medias.max(), rtol=0, atol=1e-12)]
        representante = min(candidatos, key=chave_topico)
        scores = [float(similaridades[i, j]) for i, j in combinations(membros, 2)]
        centroide = vetores[membros].mean(axis=0)
        centroide = centroide / np.linalg.norm(centroide)
        centroides.append(centroide)
        consolidados.append({
            "id": f"{prefixo}{numero:03d}",
            "label": topicos[representante]["label"],
            "description": topicos[representante]["description"],
            "representative_id": topicos[representante]["id"],
            "source_topic_ids": [topicos[i]["id"] for i in membros],
            "source_topics": [dict(topicos[i]) for i in membros],
            "cohesion_mean": float(np.mean(scores)) if scores else None,
            "cohesion_min": min(scores) if scores else None,
        })
    recebidos = [id_ for grupo in consolidados for id_ in grupo["source_topic_ids"]]
    if len(recebidos) != len(ids) or len(set(recebidos)) != len(ids) or set(recebidos) != set(ids):
        raise ValueError("A consolidação não preservou cada tópico original exatamente uma vez.")
    return consolidados, np.asarray(centroides)


def metricas_internas(topicos, consolidados, embeddings, centroides, limiar_merge, limiar_cobertura):
    antes = matriz_cosseno(embeddings, embeddings)
    depois = matriz_cosseno(centroides, centroides)
    representacao = comparar_topicos(
        topicos, consolidados, matriz_cosseno(embeddings, centroides), limiar_cobertura,
    )
    candidatos_antes = pares_internos(topicos, antes, limiar_merge)
    candidatos_depois = pares_internos(consolidados, depois, limiar_merge)
    indice = {t["id"]: i for i, t in enumerate(topicos)}
    coesoes = [
        float(antes[indice[a], indice[b]])
        for grupo in consolidados for a, b in combinations(grupo["source_topic_ids"], 2)
    ]
    recebidos = [id_ for grupo in consolidados for id_ in grupo["source_topic_ids"]]
    esperados = [t["id"] for t in topicos]
    if len(recebidos) != len(set(recebidos)) or set(recebidos) != set(esperados):
        raise ValueError("Mapeamento entre tópicos originais e consolidados inválido.")
    possiveis_antes = len(topicos) * (len(topicos) - 1) // 2
    return {
        "original_topics": len(topicos),
        "consolidated_topics": len(consolidados),
        "merged_groups": sum(len(g["source_topic_ids"]) > 1 for g in consolidados),
        "reduction_rate": 1 - len(consolidados) / len(topicos),
        "source_id_retention": len(recebidos) / len(topicos),
        "candidate_pairs_before": len(candidatos_antes),
        "candidate_pair_fraction_before": len(candidatos_antes) / possiveis_antes if possiveis_antes else 0.0,
        "candidate_pairs_after": len(candidatos_depois),
        "merged_group_mean_cosine": float(np.mean(coesoes)) if coesoes else None,
        "merged_group_min_cosine": min(coesoes) if coesoes else None,
        "semantic_coverage_original_to_consolidated": representacao["metrics"]["coverage_a_to_b"],
        "semantic_similarity_original_to_consolidated": representacao["metrics"]["similarity_a_to_b"],
        "merge_threshold": limiar_merge,
        "coverage_threshold": limiar_cobertura,
        "candidate_pairs_before_details": candidatos_antes,
        "candidate_pairs_after_details": candidatos_depois,
        "representation_metrics": representacao["metrics"],
    }


def documento_consolidado(topicos_originais, grupos, arquivo_origem, nome, metricas, contexto):
    # A saída mantém o formato groups do notebook de mesclagem e acrescenta rastreabilidade.
    return {
        "consolidation_version": "semantic_complete_link_v1",
        "approach": nome,
        "source_file": str(Path(arquivo_origem).resolve()),
        "source_sha256": hashlib.sha256(Path(arquivo_origem).read_bytes()).hexdigest(),
        "source_document": json.loads(Path(arquivo_origem).read_text(encoding="utf-8-sig")),
        "configuration": contexto,
        "internal_metrics": metricas,
        "groups": [
            {
                "group_id": g["id"], "topic_ids": g["source_topic_ids"],
                "label": g["label"], "description": g["description"],
                "representative_id": g["representative_id"],
                "source_topics": g["source_topics"],
                "cohesion_mean": g["cohesion_mean"], "cohesion_min": g["cohesion_min"],
            }
            for g in grupos
        ],
    }


def html_resumo_fluxo(relatorio, metricas_a, metricas_b, comparacao_antes):
    escapar = html.escape
    partes = [
        '<!doctype html><html lang="pt-BR"><meta charset="utf-8">',
        "<title>Consolidação e comparação</title>",
        "<style>body{font:15px system-ui;margin:24px;color:#172033}"
        "table{border-collapse:collapse;margin:16px 0}th,td{border:1px solid #ccd5e0;padding:9px}"
        "th{background:#edf2f7;text-align:left}p{max-width:1000px;line-height:1.5}</style>",
        "<h2>1. Consolidação interna, separada por abordagem</h2>",
        "<table><tr><th>Abordagem</th><th>Originais</th><th>Consolidados</th>"
        "<th>Redução</th><th>Retenção dos IDs</th><th>Cobertura semântica interna</th>"
        "<th>Coesão mínima dos grupos mesclados</th></tr>",
    ]
    for chave, dados in (("approach_a", metricas_a), ("approach_b", metricas_b)):
        coesao = dados["merged_group_min_cosine"]
        partes.append(
            f"<tr><th>{escapar(relatorio[chave]['name'])}</th>"
            f"<td>{dados['original_topics']}</td><td>{dados['consolidated_topics']}</td>"
            f"<td>{dados['reduction_rate']:.1%}</td><td>{dados['source_id_retention']:.1%}</td>"
            f"<td>{dados['semantic_coverage_original_to_consolidated']:.1%}</td>"
            f"<td>{format(coesao, '.4f') if coesao is not None else 'Sem grupos mesclados'}</td></tr>"
        )
    partes.extend([
        "</table><p>Retenção dos IDs e cobertura semântica interna referem-se aos tópicos."
        " Elas não medem a cobertura de todos os vídeos.</p>",
        "<h2>2. Comparação entre abordagens: antes e depois</h2>",
        "<table><tr><th>Etapa</th><th>Tópicos A / B</th><th>Similaridade global</th>"
        "<th>Cobertura A → B</th><th>Cobertura B → A</th><th>F1 de cobertura</th></tr>",
    ])
    for nome, dados in (("Antes do merge", comparacao_antes["metrics"]), ("Depois do merge", relatorio["metrics"])):
        partes.append(
            f"<tr><th>{nome}</th><td>{dados['topics_a']} / {dados['topics_b']}</td>"
            f"<td>{dados['global_similarity']:.4f}</td><td>{dados['coverage_a_to_b']:.1%}</td>"
            f"<td>{dados['coverage_b_to_a']:.1%}</td><td>{dados['coverage_f1']:.4f}</td></tr>"
        )
    partes.append(
        "</table><p>A similaridade é exploratória. O merge representa cada grupo por seu centroide,"
        " reduzindo a influência de tópicos repetidos. Revise os membros dos grupos nos JSONs consolidados.</p>"
        '<p><a href="comparison.html">Abrir a matriz entre as taxonomias consolidadas</a></p></html>'
    )
    return "".join(partes)


In [ ]:
from sentence_transformers import SentenceTransformer
from IPython.display import HTML, display

validar_limiar(LIMIAR_MESCLAGEM_INTERNA, "Limiar de mesclagem")
validar_limiar(LIMIAR_COBERTURA_INTERNA, "Limiar de cobertura interna")
validar_limiar(LIMIAR_SIMILARIDADE, "Limiar entre abordagens")
topicos_a = carregar_taxonomia(ARQUIVO_A)
topicos_b = carregar_taxonomia(ARQUIVO_B)
print(f"{NOME_A}: {len(topicos_a)} tópicos | {NOME_B}: {len(topicos_b)} tópicos")

modelo = SentenceTransformer(
    MODELO_EMBEDDINGS, device=DISPOSITIVO, local_files_only=SOMENTE_MODELO_LOCAL,
)
textos = [f"{t['label']}: {t['description']}" for t in topicos_a + topicos_b]
embeddings = np.asarray(modelo.encode(textos, convert_to_numpy=True, show_progress_bar=True))
corte = len(topicos_a)
emb_a, emb_b = embeddings[:corte], embeddings[corte:]

# 1. Cada taxonomia é consolidada sem consultar os tópicos da outra.
consolidados_a, centroides_a = consolidar_semanticamente(
    topicos_a, emb_a, LIMIAR_MESCLAGEM_INTERNA, prefixo="A-G",
)
consolidados_b, centroides_b = consolidar_semanticamente(
    topicos_b, emb_b, LIMIAR_MESCLAGEM_INTERNA, prefixo="B-G",
)
internas_a = metricas_internas(
    topicos_a, consolidados_a, emb_a, centroides_a,
    LIMIAR_MESCLAGEM_INTERNA, LIMIAR_COBERTURA_INTERNA,
)
internas_b = metricas_internas(
    topicos_b, consolidados_b, emb_b, centroides_b,
    LIMIAR_MESCLAGEM_INTERNA, LIMIAR_COBERTURA_INTERNA,
)

# 2. A comparação principal usa os centroides dos grupos consolidados.
comparacao_antes = comparar_topicos(
    topicos_a, topicos_b, matriz_cosseno(emb_a, emb_b), LIMIAR_SIMILARIDADE,
)
similaridades = matriz_cosseno(centroides_a, centroides_b)
relatorio = comparar_topicos(
    consolidados_a, consolidados_b, similaridades, LIMIAR_SIMILARIDADE,
)
contexto = {
    "embedding_model": MODELO_EMBEDDINGS,
    "merge_threshold": LIMIAR_MESCLAGEM_INTERNA,
    "internal_coverage_threshold": LIMIAR_COBERTURA_INTERNA,
    "external_similarity_threshold": LIMIAR_SIMILARIDADE,
    "linkage": "complete",
    "representation": "normalized_mean_of_unit_member_embeddings",
    "display_label": "medoid_original_topic",
}
relatorio.update({
    "metric_version": "bidirectional_best_cosine_v1",
    "workflow_version": "internal_merge_then_external_comparison_v1",
    "comparison_stage": "after_internal_consolidation",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "embedding_model": MODELO_EMBEDDINGS,
    "embedding_max_sequence_length": getattr(modelo, "max_seq_length", None),
    "text_fields": ["label", "description"],
    "formula": "(mean(max(C, axis=1)) + mean(max(C, axis=0))) / 2",
    "matching_policy": "independent_best_match_in_each_direction; many_to_one_allowed",
    "consolidation_configuration": contexto,
    "approach_a": {
        "name": NOME_A, "file": str(ARQUIVO_A.resolve()),
        "sha256": hashlib.sha256(ARQUIVO_A.read_bytes()).hexdigest(),
    },
    "approach_b": {
        "name": NOME_B, "file": str(ARQUIVO_B.resolve()),
        "sha256": hashlib.sha256(ARQUIVO_B.read_bytes()).hexdigest(),
    },
    "topics_a": consolidados_a, "topics_b": consolidados_b,
    "internal_metrics": {"a": internas_a, "b": internas_b},
    "comparison_before_merge": comparacao_antes,
})
arquivos = salvar_resultados(relatorio, consolidados_a, consolidados_b, similaridades, PASTA_SAIDA)

documentos = {
    "consolidated_a.json": documento_consolidado(topicos_a, consolidados_a, ARQUIVO_A, NOME_A, internas_a, contexto),
    "consolidated_b.json": documento_consolidado(topicos_b, consolidados_b, ARQUIVO_B, NOME_B, internas_b, contexto),
    "internal_metrics.json": {"configuration": contexto, "a": internas_a, "b": internas_b},
    "comparison_before_merge.json": comparacao_antes,
}
for nome, documento in documentos.items():
    caminho = PASTA_SAIDA / nome
    if caminho.resolve() in {ARQUIVO_A.resolve(), ARQUIVO_B.resolve()}:
        raise ValueError("Um destino coincide com um arquivo de entrada.")
    caminho.write_text(json.dumps(documento, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    arquivos[nome] = caminho

resumo_html = html_resumo_fluxo(relatorio, internas_a, internas_b, comparacao_antes)
caminho_resumo = PASTA_SAIDA / "workflow_report.html"
caminho_resumo.write_text(resumo_html, encoding="utf-8")
arquivos["workflow_summary"] = caminho_resumo
display(HTML(resumo_html))
display(HTML(criar_html(relatorio, consolidados_a, consolidados_b, similaridades)))

print("\nGrupos internos:")
for nome, grupos in ((NOME_A, consolidados_a), (NOME_B, consolidados_b)):
    print(nome)
    for grupo in grupos:
        print(f"  [{grupo['id']}] {grupo['label']} <- {', '.join(grupo['source_topic_ids'])}")
print("\nArquivos salvos:")
for tipo, caminho in arquivos.items():
    print(f"{tipo}: {caminho.resolve()}")
